# Extração 6 — Validação Final do Dataset

**Entregável 10**

Validação estatística e documentação do dataset de features antes de entrar no módulo de classificação.

| Passo | Análise | Critério |
|-------|---------|----------|
| 1 | Multicolinearidade (VIF) | VIF < 5 para todas as features |
| 2 | Separabilidade das classes | Índice de Fisher + curvas de densidade |
| 3 | Balanceamento | Documentar se min/max < 60% |
| 4 | Dataset final | Salvar `dataset_final.parquet` |

**Input:** `data/silver/features_selecionadas.parquet`
**Output:** `data/silver/dataset_final.parquet`

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings, time

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams.update({'figure.dpi': 100, 'font.size': 10})

In [ ]:
ROOT     = Path('..')
IN_PATH  = ROOT / 'data' / 'silver' / 'features_selecionadas.parquet'
OUT_PATH = ROOT / 'data' / 'silver' / 'dataset_final.parquet'

df = pd.read_parquet(IN_PATH)
print(f'features_selecionadas.parquet: {df.shape[0]:,} x {df.shape[1]}')

META_COLS = ['patient_id', 'task_label', 'janela_inicio_s']
meta_cols_present = [c for c in META_COLS if c in df.columns]
feat_cols = [c for c in df.columns if c not in meta_cols_present]

X_raw = df[feat_cols].values.astype(np.float64)
y     = df['task_label'].values.astype(int)
N_WIN, N_SEL = X_raw.shape
print(f'  Features : {N_SEL}  |  Janelas: {N_WIN:,}')

X = np.where(np.isnan(X_raw), 0.0, X_raw)

# Remove features com variancia zero (constantes pos-imputacao)
var_feat    = X.var(axis=0)
nonzero_var = var_feat > 1e-15
if not nonzero_var.all():
    n_drop = int((~nonzero_var).sum())
    print(f'  Removendo {n_drop} features com variancia zero...')
    X         = X[:, nonzero_var]
    feat_cols = [c for c, v in zip(feat_cols, nonzero_var) if v]
    N_SEL     = X.shape[1]

print(f'  Features para analise: {N_SEL}')
print(f'  label=0: {(y==0).sum():,}  |  label=1: {(y==1).sum():,}')

## Passo 1 — Multicolinearidade (VIF)

**VIF_j = R⁻¹[j,j]** onde R é a matriz de correlação de Pearson entre features.
Equivale a 1/(1−R²_j) da regressão de j sobre todas as demais features.

- VIF = 1: sem correlação com as demais
- VIF 1–5: correlação moderada, aceitável
- VIF > 5: correlação alta — remover iterativamente
- VIF > 10: multicolinearidade grave

A remoção iterativa remove a feature com **maior VIF** a cada passo, recalculando
o restante, até que todas fiquem abaixo de 5.

In [ ]:
def compute_vif(X_mat):
    corr = np.corrcoef(X_mat.T)
    corr = np.nan_to_num(corr, nan=0.0, posinf=1.0, neginf=-1.0)
    np.fill_diagonal(corr, 1.0)
    try:
        inv = np.linalg.inv(corr)
    except np.linalg.LinAlgError:
        inv = np.linalg.pinv(corr)
    return np.clip(np.diag(inv), 1.0, None)

print('VIF inicial:')
vif0 = compute_vif(X)
print(f'  Max VIF : {vif0.max():.2f}')
print(f'  VIF > 10: {(vif0 > 10).sum()}')
print(f'  VIF >  5: {(vif0 >  5).sum()}')
print(f'  Mediana : {np.median(vif0):.3f}')

print('\n  Top-10 VIF iniciais:')
for i in np.argsort(-vif0)[:10]:
    print(f'    VIF={vif0[i]:8.2f}  {feat_cols[i][:65]}')

In [ ]:
print('Remocao iterativa (limiar VIF = 5)...')
t0          = time.time()
keep_mask   = np.ones(N_SEL, dtype=bool)
vif_history = [float(vif0.max())]
removed_log = []

for _iter in range(N_SEL):
    kept = np.where(keep_mask)[0]
    vif_k = compute_vif(X[:, keep_mask])
    max_vif = float(vif_k.max())
    vif_history.append(max_vif)

    if max_vif <= 5.0:
        break

    worst_local  = int(np.argmax(vif_k))
    worst_global = kept[worst_local]
    removed_log.append((feat_cols[worst_global], max_vif))
    keep_mask[worst_global] = False

feat_cols_vif = [feat_cols[i] for i in np.where(keep_mask)[0]]
X_vif         = X[:, keep_mask]
N_VIF         = int(keep_mask.sum())
n_removed_vif = int((~keep_mask).sum())

print(f'  Tempo: {time.time()-t0:.1f}s')
print(f'  Features removidas: {n_removed_vif}')
print(f'  Features restantes: {N_VIF}')
print(f'  Max VIF final     : {vif_k.max():.4f}')

if removed_log:
    print('\n  Primeiras 10 features removidas (por VIF > 5):')
    for fname, vval in removed_log[:10]:
        print(f'    VIF={vval:8.2f}  {fname[:65]}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.plot(range(len(vif_history)), vif_history, 'o-', markersize=3, color='steelblue')
ax1.axhline(5,  color='orange', linestyle='--', linewidth=1.2, label='limiar = 5')
ax1.axhline(10, color='red',    linestyle='--', linewidth=1.2, label='limiar = 10')
ax1.set_xlabel('Iteracao de remocao')
ax1.set_ylabel('Max VIF')
ax1.set_title('Evolucao do VIF maximo')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

ax2 = axes[1]
ax2.hist(vif0,   bins=40, alpha=0.5, color='tomato',    label=f'Inicial  (n={N_SEL})')
ax2.hist(vif_k,  bins=40, alpha=0.7, color='steelblue', label=f'Final    (n={N_VIF})')
ax2.axvline(5, color='orange', linestyle='--', linewidth=1.2, label='VIF = 5')
ax2.set_xlabel('VIF')
ax2.set_ylabel('Contagem')
ax2.set_title('Distribuicao VIF antes e apos remocao')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(ROOT / 'data' / 'silver' / 'vif_analise.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo em data/silver/vif_analise.png')

## Passo 2 — Separabilidade Estatística das Classes

**Índice de Fisher** por feature (label=0 vs label=1):

$$FI_j = \frac{(\mu_1 - \mu_0)^2}{\sigma_0^2 + \sigma_1^2}$$

- FI próximo de 0: distribuições das duas classes se sobrepõem totalmente → feature inútil
- FI >> 1: distribuições bem separadas → feature altamente discriminativa

As curvas de densidade KDE para as 5 features com maior FI revelam se a separação é
**linear** (densidade de uma classe à direita/esquerda da outra) ou requer fronteiras não-lineares.

In [ ]:
print('Calculando indice de Fisher (label=0 vs label=1)...')
mask0, mask1 = y == 0, y == 1
n0, n1       = mask0.sum(), mask1.sum()

mu0  = X_vif[mask0].mean(axis=0)
mu1  = X_vif[mask1].mean(axis=0)
sig0 = X_vif[mask0].var(axis=0)
sig1 = X_vif[mask1].var(axis=0)

fisher = (mu1 - mu0) ** 2 / (sig0 + sig1 + 1e-12)

rank_fi   = np.argsort(-fisher)
top5_idx  = rank_fi[:5]
top10_idx = rank_fi[:10]

print(f'  Indice de Fisher — estatisticas:')
print(f'    max   : {fisher.max():.4f}')
print(f'    mediana: {np.median(fisher):.4f}')
print(f'    media top-10: {fisher[top10_idx].mean():.4f}')
print(f'    FI > 1 (boa separacao): {(fisher > 1).sum()} / {N_VIF}')

print('\n  Top-10 features por indice de Fisher:')
for k, i in enumerate(top10_idx):
    print(f'  [{k+1:2d}] FI={fisher[i]:.4f}  {feat_cols_vif[i][:65]}')

# Diagnostico de separabilidade linear
fi_mean_top10 = float(fisher[top10_idx].mean())
if fi_mean_top10 > 2.0:
    diag = 'BOA separabilidade — classificador linear deve funcionar bem'
elif fi_mean_top10 > 0.5:
    diag = 'MODERADA separabilidade — kernel RBF ou gradient boosting recomendados'
else:
    diag = 'BAIXA separabilidade — investigar features ou metodo nao-linear'
print(f'\n  Diagnostico (media FI top-10 = {fi_mean_top10:.4f}): {diag}')

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for ax, feat_i in zip(axes, top5_idx):
    fname = feat_cols_vif[feat_i]
    x0    = X_vif[mask0, feat_i]
    x1    = X_vif[mask1, feat_i]

    lo = min(x0.min(), x1.min())
    hi = max(x0.max(), x1.max())
    xi = np.linspace(lo - (hi - lo) * 0.1, hi + (hi - lo) * 0.1, 300)

    try:
        kde0 = stats.gaussian_kde(x0)
        kde1 = stats.gaussian_kde(x1)
        ax.fill_between(xi, kde0(xi), alpha=0.45, color='steelblue', label='label=0')
        ax.fill_between(xi, kde1(xi), alpha=0.45, color='tomato',    label='label=1')
        ax.plot(xi, kde0(xi), color='steelblue', linewidth=1.2)
        ax.plot(xi, kde1(xi), color='tomato',    linewidth=1.2)
    except Exception:
        ax.hist(x0, bins=30, alpha=0.5, color='steelblue', label='label=0', density=True)
        ax.hist(x1, bins=30, alpha=0.5, color='tomato',    label='label=1', density=True)

    short = fname.split('__')
    title = '__'.join(short[-2:]) if len(short) >= 2 else fname[:25]
    ax.set_title(f'{title[:30]}\nFI={fisher[feat_i]:.3f}', fontsize=8)
    ax.set_xlabel('Valor', fontsize=8)
    ax.set_ylabel('Densidade', fontsize=8)
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(ROOT / 'data' / 'silver' / 'density_top5_fisher.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Salvo em data/silver/density_top5_fisher.png')

## Passo 3 — Balanceamento do Dataset

Conta janelas válidas por paciente, task e label. Critério de desbalanceamento:

> se min(n₀, n₁) / max(n₀, n₁) < 60% → desbalanceado

**Não** aplicamos SMOTE ou undersampling aqui. A decisão pertence ao módulo de
classificação (`classificação_1.ipynb`), onde a estratégia de balanceamento pode
ser embutida dentro do fold de cross-validation (LOSO) para evitar data leakage.

In [ ]:
print('=== Balanceamento Global ===')
n0_g, n1_g = (y == 0).sum(), (y == 1).sum()
ratio_g     = min(n0_g, n1_g) / max(n0_g, n1_g)
status_g    = 'BALANCEADO' if ratio_g >= 0.60 else 'DESBALANCEADO'
print(f'  label=0: {n0_g:,}  |  label=1: {n1_g:,}  |  razao: {ratio_g:.3f}  [{status_g}]')

print('\n=== Por Paciente ===')
pid_arr   = df['patient_id'].values if 'patient_id' in df.columns else np.zeros(N_WIN, dtype=int)
patients  = sorted(set(pid_arr))
bal_rows  = []
for pid in patients:
    mask_p = pid_arr == pid
    n0_p   = int(((y == 0) & mask_p).sum())
    n1_p   = int(((y == 1) & mask_p).sum())
    total  = n0_p + n1_p
    ratio  = min(n0_p, n1_p) / max(n0_p, n1_p) if max(n0_p, n1_p) > 0 else 0.0
    flag   = ' <-- DESBALANCEADO' if ratio < 0.60 else ''
    bal_rows.append({'patient': pid, 'n_label0': n0_p, 'n_label1': n1_p,
                     'total': total, 'razao': round(ratio, 3)})
    print(f'  {pid}  n0={n0_p:5d}  n1={n1_p:5d}  total={total:5d}  razao={ratio:.3f}{flag}')

print('\n=== Por Task ===')
task_arr  = df['task_id'].values if 'task_id' in df.columns else np.zeros(N_WIN, dtype=int)
tasks     = sorted(set(task_arr))
task_rows = []
for task in tasks:
    mask_t = task_arr == task
    n0_t   = int(((y == 0) & mask_t).sum())
    n1_t   = int(((y == 1) & mask_t).sum())
    total  = n0_t + n1_t
    ratio  = min(n0_t, n1_t) / max(n0_t, n1_t) if max(n0_t, n1_t) > 0 else 0.0
    flag   = ' <-- DESBALANCEADO' if ratio < 0.60 else ''
    task_rows.append({'task': task, 'n_label0': n0_t, 'n_label1': n1_t,
                      'total': total, 'razao': round(ratio, 3)})
    print(f'  {str(task):<12s}  n0={n0_t:5d}  n1={n1_t:5d}  total={total:5d}  razao={ratio:.3f}{flag}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
df_bal = pd.DataFrame(bal_rows).set_index('patient')
df_bal[['n_label0', 'n_label1']].plot(kind='bar', ax=ax1,
    color=['steelblue', 'tomato'], alpha=0.8)
ax1.set_title('Janelas por Paciente e Label')
ax1.set_xlabel('Paciente')
ax1.set_ylabel('Contagem')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(axis='y', alpha=0.3)
ax1.legend(['label=0', 'label=1'])

ax2 = axes[1]
df_task = pd.DataFrame(task_rows).set_index('task')
df_task[['n_label0', 'n_label1']].plot(kind='bar', ax=ax2,
    color=['steelblue', 'tomato'], alpha=0.8)
ax2.set_title('Janelas por Task e Label')
ax2.set_xlabel('Task')
ax2.set_ylabel('Contagem')
ax2.tick_params(axis='x', rotation=45)
ax2.grid(axis='y', alpha=0.3)
ax2.legend(['label=0', 'label=1'])

plt.tight_layout()
plt.savefig(ROOT / 'data' / 'silver' / 'balance_analysis.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Salvo em data/silver/balance_analysis.png')

## Passo 4 — Dataset Final

Gera `dataset_final.parquet` com as features após VIF + metadados.
Documenta as estatísticas finais do pipeline de extração completo.

**Critério de correlação inter-features:** média < 0.5 (em valor absoluto) após a seleção.

In [ ]:
print('Calculando correlacao media inter-features...')
# Usa amostra para velocidade se N_VIF > 200
N_CORR = min(N_VIF, 200)
idx_corr = np.argsort(-fisher[:N_VIF])[:N_CORR]  # top features por Fisher
X_corr = X_vif[:, idx_corr]

corr_mat   = np.abs(np.corrcoef(X_corr.T))
np.fill_diagonal(corr_mat, 0.0)
triu_idx   = np.triu_indices(N_CORR, k=1)
mean_corr  = float(corr_mat[triu_idx].mean())
max_corr   = float(corr_mat[triu_idx].max())
corr_ok    = 'OK' if mean_corr < 0.50 else 'ATENCAO: media > 0.5'

print(f'  Correlacao Pearson |r| media  : {mean_corr:.4f}  [{corr_ok}]')
print(f'  Correlacao Pearson |r| maxima : {max_corr:.4f}')

# Heatmap (amostra)
N_HEAT = min(40, N_CORR)
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(corr_mat[:N_HEAT, :N_HEAT], vmin=0, vmax=1,
               cmap='RdYlGn_r', aspect='auto')
plt.colorbar(im, ax=ax, label='|Pearson r|')
ax.set_title(f'Correlacao entre top-{N_HEAT} features (por Fisher)')
ax.set_xlabel('Feature')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.savefig(ROOT / 'data' / 'silver' / 'feature_correlation_heatmap.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Salvo em data/silver/feature_correlation_heatmap.png')

print('\n=== Estatisticas Finais do Pipeline ===')
print(f'  Pacientes          : {len(patients)}')
print(f'  Janelas totais     : {N_WIN:,}')
print(f'  Janelas label=0    : {n0_g:,}  ({100*n0_g/N_WIN:.1f}%)')
print(f'  Janelas label=1    : {n1_g:,}  ({100*n1_g/N_WIN:.1f}%)')
print(f'  Razao classes      : {ratio_g:.3f}  [{status_g}]')
print(f'  Features (entrada) : {N_SEL}  (features_selecionadas.parquet)')
print(f'  Features (apos VIF): {N_VIF}')
print(f'  Removidas por VIF  : {n_removed_vif}')
print(f'  Corr. media |r|    : {mean_corr:.4f}  [{corr_ok}]')
print(f'  FI medio top-10    : {fi_mean_top10:.4f}  ({diag.split(" — ")[0]})')

In [ ]:
print(f'Salvando dataset_final.parquet ({N_VIF} features)...')
df_final = pd.DataFrame(X_vif.astype(np.float32), columns=feat_cols_vif)
for col in meta_cols_present:
    df_final[col] = df[col].values

df_final.to_parquet(OUT_PATH, index=False)
size_mb = OUT_PATH.stat().st_size / 1e6
print(f'  Salvo : {OUT_PATH}')
print(f'  Shape : {df_final.shape[0]:,} x {df_final.shape[1]}')
print(f'  Tamanho: {size_mb:.1f} MB')

# Manifesto do pipeline completo
manifest = {
    'n_patients'       : len(patients),
    'n_windows'        : int(N_WIN),
    'n_windows_label0' : int(n0_g),
    'n_windows_label1' : int(n1_g),
    'class_ratio'      : round(ratio_g, 4),
    'n_features_raw'   : int(N_SEL),
    'n_features_final' : int(N_VIF),
    'n_removed_vif'    : int(n_removed_vif),
    'mean_inter_corr'  : round(mean_corr, 4),
    'fi_mean_top10'    : round(fi_mean_top10, 4),
}
import json as _json
manifest_path = ROOT / 'data' / 'silver' / 'dataset_manifest.json'
manifest_path.write_text(_json.dumps(manifest, indent=2))
print(f'  Manifesto: {manifest_path}')
print()
print(_json.dumps(manifest, indent=2))

## Resumo — Entregável 10

| Item | Resultado |
|------|-----------|
| Features de entrada | `features_selecionadas.parquet` |
| Features após VIF < 5 | `N_VIF` |
| Max VIF final | < 5.0 |
| Separabilidade (FI top-10) | ver `fi_mean_top10` |
| Balanceamento global | ver `ratio_g` |
| Correlação inter-features | ver `mean_corr` |
| Saída | `dataset_final.parquet` + `dataset_manifest.json` |

### Pipeline completo de Extração

```
PP (Entregáveis 1–5)
  └─ segmentação           → windows_metadata.parquet
extração_1_segmentação     → X (N_WIN × N_CH × 500)
extração_2_features        → features_raw.parquet         (Entregável 6)
extração_3_feature_eng.    → features_engineered.parquet  (Entregável 7)
extração_4_dim_reduction   → features_pca.parquet         (Entregável 8)
extração_5_selecao_attr.   → features_selecionadas.parquet(Entregável 9)
extração_6_validacao_final → dataset_final.parquet        (Entregável 10)
  └─ classificação_1.ipynb → LOSO (RF / XGBoost / SVM)
```

**Próximo:** `classificação_1.ipynb` — Leave-One-Subject-Out cross-validation